In [26]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from tinyshift.modelling import fourier_seasonality
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression


In [27]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# LigthGBM

In [28]:

#def create_mlforecast_multiquantile():
#        import lightgbm as lgb
#
#     # Define 90% and 50% prediction interval quantile LightGBM models
#     models = {
#         "LGBM-lo-90": lgb.LGBMRegressor(objective="quantile", alpha=0.05, random_state=42),
#         "LGBM-hi-90": lgb.LGBMRegressor(objective="quantile", alpha=0.95, random_state=42),
#         "LGBM-lo-50": lgb.LGBMRegressor(objective="quantile", alpha=0.25, random_state=42),
#         "LGBM-hi-50": lgb.LGBMRegressor(objective="quantile", alpha=0.75, random_state=42),
#     }
#     return MLForecast(
#         models=models,
#         freq="MS",
#         lags=[1, 7],
#     )
#models = create_mlforecast_multiquantile()

# RandomForestQuantileRegressor

In [29]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [30]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-lo-50": QuantileRF(
            quantile=0.25, n_estimators=100, max_depth=8, random_state=42
        ),
        "RF-hi-50": QuantileRF(
            quantile=0.75, n_estimators=100, max_depth=8, random_state=42
        ),
        "LinearRegression": LinearRegression()
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

In [31]:
cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     quantile_cols=[
         ("RF-lo-90", "RF-hi-90"),
         ("RF-lo-50", "RF-hi-50"),
     ],
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,quantile_cols,"[('RF-lo-90', ...), ('RF-lo-50', ...)]"
,n_windows,7
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [32]:
cqr.predict_interval(h=7, X_df=test)

,unique_id,ds,RF-lo-90,RF-hi-90,RF-lo-50,RF-hi-50,LinearRegression,RF-lo-90-CQR,RF-hi-90-CQR,RF-lo-50-CQR,RF-hi-50-CQR
0,1,1960-01-01,317.7,407.0,342.0,405.0,414.916253,297.70,427.00,311.0,436.0
1,1,1960-02-01,301.0,420.0,318.0,405.0,442.818017,299.00,422.00,310.0,413.0
2,1,1960-03-01,277.0,548.0,348.0,406.0,481.009648,233.05,591.95,304.0,450.0
3,1,1960-04-01,259.0,559.0,348.0,406.0,498.236029,254.00,564.00,306.0,448.0
4,1,1960-05-01,233.0,559.0,355.0,420.0,501.106592,246.00,546.00,319.0,456.0
5,1,1960-06-01,228.9,559.0,363.0,491.0,492.419196,215.90,572.00,312.0,542.0
6,1,1960-07-01,203.0,559.0,355.0,548.0,492.083112,141.00,621.00,293.0,610.0


In [33]:
cqr.evaluate(test, h=12, alpha=0.05)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
1,RF,50%,0.05,0.083,117.646,1590.146
0,RF,90%,0.05,0.750,323.617,723.617
3,RF-CQR,50%,0.05,0.583,203.396,378.396
2,RF-CQR,90%,0.05,0.917,342.792,346.125


# GradientBoostingRegressor & HistGradientBoostingRegressor

In [34]:
def model_callable():
    models = {
        "GBR-lo-90": GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=100, random_state=42),
        "GBR-hi-90": GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=100, random_state=42),
        "HGB-lo-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.05, max_iter=100, random_state=42),
        "HGB-hi-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.95, max_iter=100, random_state=42),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )

models = model_callable()

In [35]:
cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     quantile_cols=[
         ("GBR-lo-90", "GBR-hi-90"),
         ("HGB-lo-90", "HGB-hi-90"),
     ],
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,quantile_cols,"[('GBR-lo-90', ...), ('HGB-lo-90', ...)]"
,n_windows,7
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [36]:
cqr.predict_interval(h=7, X_df=test)

,unique_id,ds,GBR-lo-90,GBR-hi-90,HGB-lo-90,HGB-hi-90,GBR-lo-90-CQR,GBR-hi-90-CQR,HGB-lo-90-CQR,HGB-hi-90-CQR
0,1,1960-01-01,335.188588,416.279655,367.308207,530.362794,304.355685,447.112557,365.314188,532.356812
1,1,1960-02-01,311.057873,421.348784,373.442907,536.517463,298.354314,434.052343,371.448889,538.511481
2,1,1960-03-01,296.880283,455.205795,371.814552,539.515977,247.027623,505.058455,329.820534,581.509995
3,1,1960-04-01,292.948881,494.575852,369.074321,542.999380,252.504072,535.020661,370.201920,541.871781
4,1,1960-05-01,292.948881,543.177811,370.702676,542.961234,254.118262,582.008431,377.830275,535.833635
5,1,1960-06-01,292.007464,543.856854,378.605889,545.091961,231.583624,604.280693,352.494878,571.202972
6,1,1960-07-01,290.825417,545.876043,376.335765,545.137402,204.488433,632.213027,307.630097,613.843070


In [43]:
cqr.evaluate(test, h=12, alpha=0.05)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,GBR,90%,0.05,0.667,220.468,772.854
2,GBR-CQR,90%,0.05,1.000,307.241,307.241
1,HGB,90%,0.05,0.833,197.552,661.604
3,HGB-CQR,90%,0.05,0.917,236.552,263.742
